In [0]:
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
# Drop all silver tables so they are recreated with correct data types
for t in ["region", "nation", "airline", "route", "airline_route", "passenger", "booking", "flight_booking"]:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.{t}")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.{t}_quarantine")
spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.data_quality_report")
print("dropped all silver tables")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

DATE_MIN, DATE_MAX = F.lit("1994-01-01").cast("date"), F.lit("1998-12-31").cast("date")
dq_report_rows = []


def validate_and_write(table, df, checks):
    """checks: (rule_code, description, fail_condition TRUE when violated). Silver gets clean data columns only; quarantine adds rejection_reason."""
    data_cols = [c for c in df.columns if not c.startswith("_") and c != "pipeline_runID"]
    fcols = [f"_f_{k}" for k, _d, _c in checks]
    reasons = F.filter(F.array(*[F.when(c, F.lit(k)) for k, _d, c in checks]), lambda x: x.isNotNull())
    flagged = df.select(
        "*",
        *[c.alias(f) for (_k, _d, c), f in zip(checks, fcols)],
        F.when(F.size(reasons) > 0, F.array_join(reasons, ", ")).alias("rejection_reason"),
    )

    counts = flagged.agg(
        F.count(F.lit(1)).alias("_total"),
        F.sum(F.col("rejection_reason").isNotNull().cast("int")).alias("_quarantined"),
        *[F.sum(F.col(f"_f_{k}").cast("int")).alias(k) for k, _d, _c in checks],
    ).collect()[0].asDict()

    for k, d, _c in checks:
        n = int(counts[k] or 0)
        dq_report_rows.append((table, k, d, n, f"quarantined to {catalog}.silver.{table}_quarantine" if n else "passed"))

    flagged.where("rejection_reason IS NULL").select(*data_cols).write.mode("overwrite").saveAsTable(f"{catalog}.silver.{table}")
    flagged.where("rejection_reason IS NOT NULL").select(*data_cols, "rejection_reason").write.mode("overwrite").saveAsTable(f"{catalog}.silver.{table}_quarantine")

    print(f"{table}: {counts['_total']} rows in, {int(counts['_quarantined'] or 0)} quarantined")

In [0]:
# region: PK not null + unique
region_df = spark.read.table(f"{catalog}.bronze.region")

clean_region = region_df.select("r_regionkey", "r_name", "r_comment")

validate_and_write("region", clean_region, [
    ("NULL_PK", "r_regionkey not null", F.col("r_regionkey").isNull()),
    ("DUP_PK", "r_regionkey unique", F.count("r_regionkey").over(Window.partitionBy("r_regionkey")) > 1),
])

In [0]:
# nation: PK not null + unique
nation_df = spark.read.table(f"{catalog}.bronze.nation")

clean_nation = nation_df.select("n_nationkey", "n_name", "n_regionkey", "n_comment")

validate_and_write("nation", clean_nation, [
    ("NULL_PK", "n_nationkey not null", F.col("n_nationkey").isNull()),
    ("DUP_PK", "n_nationkey unique", F.count("n_nationkey").over(Window.partitionBy("n_nationkey")) > 1),
])

In [0]:
# airline: cast keys/rating, FK al_nationkey -> nation
airline_df = spark.read.table(f"{catalog}.bronze.airline")

clean_airline = airline_df.select(
    F.col("al_airlineid").cast("int"), "al_name", "al_address", F.col("al_nationkey").cast("int"),
    "al_phone", F.col("al_rating").cast("double"), "al_comment",
).join(
    spark.table(f"{catalog}.silver.nation").select(F.col("n_nationkey").alias("_fk")),
    F.col("al_nationkey") == F.col("_fk"), "left",
)

validate_and_write("airline", clean_airline, [
    ("NULL_PK", "al_airlineid not null", F.col("al_airlineid").isNull()),
    ("DUP_PK", "al_airlineid unique", F.count("al_airlineid").over(Window.partitionBy("al_airlineid")) > 1),
    ("INVALID_FK_NATION", "al_nationkey exists in silver.nation", F.col("_fk").isNull()),
])

In [0]:
# route: cast id/capacity/baseprice, PK + baseprice > 0 checks
route_df = spark.read.table(f"{catalog}.bronze.route")

clean_route = route_df.select(
    F.col("ro_routeid").cast("int"), "ro_name", "ro_aircraft", "ro_airline_brand", "ro_routetype",
    F.col("ro_capacity").cast("int"), "ro_cabinconfig", F.col("ro_baseprice").cast("double"), "ro_comment",
)

validate_and_write("route", clean_route, [
    ("NULL_PK", "ro_routeid not null", F.col("ro_routeid").isNull()),
    ("DUP_PK", "ro_routeid unique", F.count("ro_routeid").over(Window.partitionBy("ro_routeid")) > 1),
    ("NUM_BASEPRICE_NONPOSITIVE", "ro_baseprice > 0", F.col("ro_baseprice").isNull() | (F.col("ro_baseprice") <= 0)),
])

In [0]:
# airline_route: cast ids/seats/cost, PK + operatingcost check, FKs to airline/route
airline_route_df = spark.read.table(f"{catalog}.bronze.airline_route")

clean_airline_route = airline_route_df.select(
    F.col("ar_airlineid").cast("int"), F.col("ar_routeid").cast("int"),
    F.col("ar_available_seats").cast("int"), F.col("ar_operatingcost").cast("double"), "ar_comment",
).join(
    spark.table(f"{catalog}.silver.airline").select(F.col("al_airlineid").alias("_fk_airline")),
    F.col("ar_airlineid") == F.col("_fk_airline"), "left",
).join(
    spark.table(f"{catalog}.silver.route").select(F.col("ro_routeid").alias("_fk_route")),
    F.col("ar_routeid") == F.col("_fk_route"), "left",
)

validate_and_write("airline_route", clean_airline_route, [
    ("NULL_PK", "ar_airlineid and ar_routeid not null", F.col("ar_airlineid").isNull() | F.col("ar_routeid").isNull()),
    ("DUP_PK", "(ar_airlineid, ar_routeid) unique", F.count(F.lit(1)).over(Window.partitionBy("ar_airlineid", "ar_routeid")) > 1),
    ("NUM_OPERATINGCOST_NONPOSITIVE", "ar_operatingcost > 0", F.col("ar_operatingcost").isNull() | (F.col("ar_operatingcost") <= 0)),
    ("INVALID_FK_AIRLINE", "ar_airlineid exists in silver.airline", F.col("_fk_airline").isNull()),
    ("INVALID_FK_ROUTE", "ar_routeid exists in silver.route", F.col("_fk_route").isNull()),
])

In [0]:
# passenger: cast id/nationkey/loyaltypoints, PK + name completeness, FK to nation
passenger_df = spark.read.table(f"{catalog}.bronze.passenger")

clean_passenger = passenger_df.select(
    F.col("pa_passengerid").cast("int"), "pa_name", "pa_address", F.col("pa_nationkey").cast("int"),
    "pa_phone", F.col("pa_loyaltypoints").cast("double"), "pa_travelpurpose", "pa_comment",
).join(
    spark.table(f"{catalog}.silver.nation").select(F.col("n_nationkey").alias("_fk")),
    F.col("pa_nationkey") == F.col("_fk"), "left",
)

validate_and_write("passenger", clean_passenger, [
    ("NULL_PK", "pa_passengerid not null", F.col("pa_passengerid").isNull()),
    ("DUP_PK", "pa_passengerid unique", F.count("pa_passengerid").over(Window.partitionBy("pa_passengerid")) > 1),
    ("NULL_NAME", "pa_name not null", F.col("pa_name").isNull()),
    ("INVALID_FK_NATION", "pa_nationkey exists in silver.nation", F.col("_fk").isNull()),
])

In [0]:
# booking: PK, totalprice > 0, date window + flightdate >= bookingdate, FK to passenger
booking_df = spark.read.table(f"{catalog}.bronze.booking")

clean_booking = booking_df.select(
    "bk_bookingid", "bk_passengerid", "bk_status", "bk_totalprice", "bk_bookingdate", "bk_flightdate",
    "bk_bookingpriority", "bk_agent", "bk_comment",
).join(
    spark.table(f"{catalog}.silver.passenger").select(F.col("pa_passengerid").alias("_fk")),
    F.col("bk_passengerid") == F.col("_fk"), "left",
)

validate_and_write("booking", clean_booking, [
    ("NULL_PK", "bk_bookingid not null", F.col("bk_bookingid").isNull()),
    ("DUP_PK", "bk_bookingid unique", F.count("bk_bookingid").over(Window.partitionBy("bk_bookingid")) > 1),
    ("NUM_TOTALPRICE_NONPOSITIVE", "bk_totalprice > 0", F.col("bk_totalprice").isNull() | (F.col("bk_totalprice") <= 0)),
    ("DATE_WINDOW_BK_BOOKINGDATE", "bk_bookingdate in [1994, 1998]", F.col("bk_bookingdate").isNull() | ~F.col("bk_bookingdate").between(DATE_MIN, DATE_MAX)),
    ("DATE_WINDOW_BK_FLIGHTDATE", "bk_flightdate in [1994, 1998]", F.col("bk_flightdate").isNull() | ~F.col("bk_flightdate").between(DATE_MIN, DATE_MAX)),
    ("DATE_FLIGHT_BEFORE_BOOKING", "bk_flightdate >= bk_bookingdate", F.col("bk_flightdate") < F.col("bk_bookingdate")),
    ("INVALID_FK_PASSENGER", "bk_passengerid exists in silver.passenger", F.col("_fk").isNull()),
])

In [0]:
# flight_booking: PK, number checks, date window + cross-column date checks, FKs to booking/route/airline
flight_booking_df = spark.read.table(f"{catalog}.bronze.flight_booking")

clean_flight_booking = flight_booking_df.select(
    "fb_bookingid", "fb_routeid", "fb_airlineid", "fb_linenumber", "fb_seats", "fb_ticketprice",
    "fb_discount", "fb_tax", "fb_refundflag", "fb_segmentstatus", "fb_departuredate", "fb_commitdate",
    "fb_arrivaldate", "fb_cabinclass", "fb_comment",
).join(
    spark.table(f"{catalog}.silver.booking").select(F.col("bk_bookingid").alias("_bk_id"), F.col("bk_bookingdate").alias("_bk_date")),
    F.col("fb_bookingid") == F.col("_bk_id"), "left",
).join(
    spark.table(f"{catalog}.silver.route").select(F.col("ro_routeid").alias("_fk_route")),
    F.col("fb_routeid") == F.col("_fk_route"), "left",
).join(
    spark.table(f"{catalog}.silver.airline").select(F.col("al_airlineid").alias("_fk_airline")),
    F.col("fb_airlineid") == F.col("_fk_airline"), "left",
)

validate_and_write("flight_booking", clean_flight_booking, [
    ("NULL_PK", "fb_bookingid and fb_linenumber not null", F.col("fb_bookingid").isNull() | F.col("fb_linenumber").isNull()),
    ("DUP_PK", "(fb_bookingid, fb_linenumber) unique", F.count(F.lit(1)).over(Window.partitionBy("fb_bookingid", "fb_linenumber")) > 1),
    ("NUM_TICKETPRICE_NONPOSITIVE", "fb_ticketprice > 0", F.col("fb_ticketprice").isNull() | (F.col("fb_ticketprice") <= 0)),
    ("NUM_DISCOUNT_OUT_OF_RANGE", "fb_discount in [0, 1]", F.col("fb_discount").isNull() | ~F.col("fb_discount").between(0, 1)),
    ("NUM_SEATS_NONPOSITIVE", "fb_seats >= 1", F.col("fb_seats").isNull() | (F.col("fb_seats") < 1)),
    ("DATE_WINDOW_FB_DEPARTUREDATE", "fb_departuredate in [1994, 1998]", F.col("fb_departuredate").isNull() | ~F.col("fb_departuredate").between(DATE_MIN, DATE_MAX)),
    ("DATE_WINDOW_FB_COMMITDATE", "fb_commitdate in [1994, 1998]", F.col("fb_commitdate").isNull() | ~F.col("fb_commitdate").between(DATE_MIN, DATE_MAX)),
    ("DATE_WINDOW_FB_ARRIVALDATE", "fb_arrivaldate in [1994, 1998]", F.col("fb_arrivaldate").isNull() | ~F.col("fb_arrivaldate").between(DATE_MIN, DATE_MAX)),
    ("DATE_DEPARTURE_BEFORE_BOOKING", "fb_departuredate >= bk_bookingdate", F.col("_bk_date").isNotNull() & (F.col("fb_departuredate") < F.col("_bk_date"))),
    ("DATE_ARRIVAL_BEFORE_DEPARTURE", "fb_arrivaldate >= fb_departuredate", F.col("fb_arrivaldate") < F.col("fb_departuredate")),
    ("INVALID_FK_BOOKING", "fb_bookingid exists in silver.booking", F.col("_bk_id").isNull()),
    ("INVALID_FK_ROUTE", "fb_routeid exists in silver.route", F.col("_fk_route").isNull()),
    ("INVALID_FK_AIRLINE", "fb_airlineid exists in silver.airline", F.col("_fk_airline").isNull()),
])

In [0]:
# Bronze = Silver + Quarantine for every table
rows = []
for t in ["region", "nation", "airline", "route", "airline_route", "passenger", "booking", "flight_booking"]:
    b = spark.table(f"{catalog}.bronze.{t}").count()
    s = spark.table(f"{catalog}.silver.{t}").count()
    q = spark.table(f"{catalog}.silver.{t}_quarantine").count()
    rows.append((t, b, s, q, b == s + q))

display(spark.createDataFrame(rows, "table string, bronze long, silver long, quarantine long, balanced boolean"))
assert all(r[4] for r in rows), f"reconciliation failed: {[r[0] for r in rows if not r[4]]}"

In [0]:
# Data-quality report: every rule, every table (zero-failure rules included)
dq_report = spark.createDataFrame(
    dq_report_rows, "table string, rule_code string, check_description string, failing_rows long, action_taken string"
).orderBy("table", "rule_code")
dq_report.write.mode("overwrite").saveAsTable(f"{catalog}.silver.data_quality_report")
display(dq_report)